# Translate Comparison Marc

Research code accompanying [Şeker (2026), Expert Systems](https://doi.org/10.1111/exsy.70300).
See the [repository README](../README.md) for dependencies, dataset preparation and citation.

This is an exploratory notebook: select the intended dataset and experiment cells before
running. Local intermediate datasets and manual notebook state are required. Stored
outputs and execution counts have been removed. Model and API identifiers retain the
historical experiment settings; check availability before calling external services.

Both historical translation notebooks currently load `data/MARC_translate_sentiment_topic_all`; the M-ABSA filename does not establish a separate M-ABSA experiment.


In [ ]:
from pathlib import Path
import os

# Keep relative data/model paths anchored at the repository root.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)


In [ ]:
%pip install sacremoses
%pip install -U "transformers>=4.44" sentencepiece accelerate "huggingface_hub[hf_transfer]"

In [ ]:
import pandas as pd
import requests
from tqdm import tqdm

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("data/MARC_translate_sentiment_topic_all")
df = dataset.to_pandas()

In [ ]:
df = df.rename(columns={"translated_text": "translated_gemini"})

In [ ]:
sampled_df = df[df['language'] != 'en'].groupby('language').apply(lambda x: x.sample(n=100, random_state=42)).reset_index(drop=True)

In [ ]:
%pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Set the Deepl API key
DEEPL_API_KEY = os.getenv("DEEPL_API_KEY")

In [ ]:
import requests

def deepl_translate(text, source_lang="auto", target_lang="en"):
    url = "https://api-free.deepl.com/v2/translate"
    headers = {
        "Authorization": f"DeepL-Auth-Key {DEEPL_API_KEY}"
    }
    params = {
        "text": text,
        "source_lang": source_lang.upper() if source_lang != "auto" else None,
        "target_lang": target_lang.upper()
    }

    try:
        response = requests.post(url, headers=headers, data=params)
        if response.status_code == 200:
            return response.json()["translations"][0]["text"]
        else:
            return f"ERROR {response.status_code}: {response.text}"
    except Exception as e:
        return f"ERROR: {str(e)}"


In [ ]:
from tqdm import tqdm
import time

def run_deepl_batched_translation(df: pd.DataFrame, text_col: str = "text", lang_col: str = "language") -> pd.DataFrame:
    deepl_translations = []
    deepl_errors = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="DeepL Translation"):
        text = row[text_col]
        source_lang = row[lang_col]

        result = deepl_translate(text, source_lang=source_lang, target_lang="en")

        if result.startswith("ERROR"):
            deepl_translations.append(None)
            deepl_errors.append(result)
        else:
            deepl_translations.append(result)
            deepl_errors.append(None)

        time.sleep(1.1)  # 🔄 60 istek/dakika sınırı (free plan için güvenli)

    df["translated_deepl"] = deepl_translations
    df["deepl_error"] = deepl_errors

    return df


In [ ]:
translated_df = run_deepl_batched_translation(sampled_df)

In [ ]:
%pip install transformers sentencepiece

In [ ]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# Model ve tokenizer yükle
model_name = "facebook/m2m100_418M"
tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

def m2m_translate(text: str, source_lang: str, target_lang: str = "en") -> str:
    try:
        tokenizer.src_lang = source_lang
        encoded = tokenizer(text, return_tensors="pt")
        generated_tokens = model.generate(
            **encoded,
            forced_bos_token_id=tokenizer.get_lang_id(target_lang),
            max_length=512
        )
        return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    except Exception as e:
        return f"❌ ERROR: {e}"


In [ ]:
example = "J'aime ce produit, mais la livraison était lente."
translated = m2m_translate(example, source_lang="fr", target_lang="en")
print(translated)

In [ ]:
# M2M100 örneği
model.save_pretrained("models/m2m100_418M")
tokenizer.save_pretrained("models/m2m100_418M")

In [ ]:
from tqdm import tqdm

def apply_m2m_translation(df, text_col="text", lang_col="language"):
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Translating with M2M100"):
        text = row[text_col]
        src_lang = row[lang_col]
        if src_lang == "en":
            results.append(text)  # İngilizce ise olduğu gibi al
        else:
            translated = m2m_translate(text, source_lang=src_lang, target_lang="en")
            results.append(translated)
    return results

In [ ]:
sampled_df["translated_m2m"] = apply_m2m_translation(sampled_df)

In [ ]:
# Gerekenler:
# pip install "transformers>=4.44" sentencepiece accelerate "huggingface_hub[hf_transfer]"
# (opsiyonel hız) export HF_HUB_ENABLE_HF_TRANSFER=1

import math
import sys, traceback
import pandas as pd
from collections import defaultdict
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = "facebook/nllb-200-distilled-600M"
TGT_LANG = "eng_Latn"
BATCH_SIZE = 8          # CPU için makul, arttırılabilir
MAX_NEW_TOKENS = 200

# 2-harf -> NLLB kodu
ISO2_TO_NLLB = {
    "ar": "arb_Arab",
    "da": "dan_Latn",
    "de": "deu_Latn",
    "en": "eng_Latn",
    "es": "spa_Latn",
    "fr": "fra_Latn",
    "hi": "hin_Deva",
    "hr": "hrv_Latn",
    "id": "ind_Latn",
    "ja": "jpn_Jpan",
    "ko": "kor_Hang",
    "nl": "nld_Latn",
    "pt": "por_Latn",
    "ru": "rus_Cyrl",
    "sk": "slk_Latn",
    "sv": "swe_Latn",
    "sw": "swh_Latn",
    "th": "tha_Thai",
    "tr": "tur_Latn",
    "vi": "vie_Latn",
    "zh": "zho_Hans",   # gerekirse zho_Hant'a değiştirin
}

# Model + tokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

translator = pipeline(
    "translation",
    model=mdl,
    tokenizer=tok,
    max_length=400,
    max_new_tokens=MAX_NEW_TOKENS,
    num_beams=1,        # hız
    do_sample=False,
)

def map_iso2_to_nllb(codes):
    out, missing = [], []
    for c in codes:
        n = ISO2_TO_NLLB.get(c)
        if n is None:
            missing.append(c)
        out.append(n)
    if missing:
        print("[WARN] NLLB eşlemesi bulunamadı:", sorted(set(missing)))
    return out

def translate_df_to_english(df, text_col="text", lang_col="language", out_col="translation_en_nllb"):
    # Çıkış kolonu oluştur/temizle
    if out_col not in df.columns:
        df[out_col] = None

    # NaN/boşları ele
    mask_valid = df[text_col].notna() & df[text_col].astype(str).str.len().gt(0) & df[lang_col].notna()
    idx_valid = df.index[mask_valid]

    # İngilizce olanları kopyala (kayıp yoksa hız kazanır)
    mask_en = mask_valid & (df[lang_col].str.lower() == "en")
    df.loc[mask_en, out_col] = df.loc[mask_en, text_col]

    # İngilizce dışındakiler
    mask_non_en = mask_valid & (df[lang_col].str.lower() != "en")
    idxs = df.index[mask_non_en]
    if len(idxs) == 0:
        return df

    src_iso2 = df.loc[idxs, lang_col].str.lower().tolist()
    src_nllb = map_iso2_to_nllb(src_iso2)
    texts = df.loc[idxs, text_col].tolist()

    # Kaynak dil koduna göre grupla
    groups = defaultdict(list)
    for ridx, t, s_nllb in zip(idxs, texts, src_nllb):
        groups[s_nllb].append((ridx, t))

    # Çeviri
    for s_lang, items in groups.items():
        if s_lang is None:
            # Eşleşmeyen dil; None bırakıyoruz
            continue
        row_ids, batch_texts = zip(*items)

        try:
            outs = translator(
                list(batch_texts),
                src_lang=s_lang,     # 🔑 dil burada veriliyor
                tgt_lang=TGT_LANG,
                batch_size=BATCH_SIZE,
                truncation=True,
            )
            for r, o in zip(row_ids, outs):
                df.at[r, out_col] = o["translation_text"]
        except Exception as e:
            print(f"[GROUP ERROR] src_lang={s_lang}: {e}", file=sys.stderr)
            traceback.print_exc()
            # Tek tek dene ki sorunlu kayıtları yakalayalım
            for r, txt in items:
                try:
                    o = translator(txt, src_lang=s_lang, tgt_lang=TGT_LANG, truncation=True)[0]["translation_text"]
                    df.at[r, out_col] = o
                except Exception as e2:
                    print(f"[ITEM ERROR] idx={r}, src={s_lang}: {e2}", file=sys.stderr)
                    traceback.print_exc()
                    df.at[r, out_col] = None
    return df
dfx = df_sampled.head(10)

df = translate_df_to_english(dfx, text_col="text", lang_col="language", out_col="translated_nllb")

In [ ]:
print(df_marc["translated_nllb"].notna().sum(), "satır çevrildi /", len(df_marc))

In [ ]:
# Kaynak metin uzunluğu / çeviri uzunluğu oranı
src_len = df_marc['text'].astype(str).str.len().clip(lower=1)
tgt_len = df_marc['translated_nllb'].astype(str).str.len().clip(lower=1)
df_marc['len_ratio_en'] = tgt_len / src_len

# Çok düşük oranlı (ör. <0.35) olanları incelemeye al
suspect = df_marc[df_marc['len_ratio_en'] < 0.35].copy()
print("Şüpheli (kısalmış olabilecek) çeviri sayısı:", len(suspect))
suspect[['language','text']].head(2)


In [ ]:
keep_cols = [
    "language", 
    "text",
    "translated_deepl",
    "translated_nllb",    # yeni NLLB çeviriniz
    "translated_gemini",        # varsa önceki sisteminiz
    "translated_m2m",
]

df = df_marc[keep_cols].copy()

print("Yeni shape:", df.shape)
df.head(3)

In [ ]:
# save parquet df
df.to_parquet('translated_sampled_dataset_marc.parquet', index=False)

In [ ]:
def tfidf_similarity(a, b):
    try:
        vect = TfidfVectorizer().fit([a, b])
        tfidf_matrix = vect.transform([a, b])
        return cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    except:
        return None

In [ ]:
def closest_to_deepl(row):
    g_sim = tfidf_similarity(row['translated_text'], row['translated_deepl']) if pd.notnull(row['translated_text']) else -1
    h_sim = tfidf_similarity(row['helsinki_translation'], row['translated_deepl']) if pd.notnull(row['helsinki_translation']) else -1

    if g_sim > h_sim:
        return 'gemini'
    elif h_sim > g_sim:
        return 'helsinki'
    elif g_sim == h_sim and g_sim != -1:
        return 'equal'
    else:
        return 'unknown'

In [ ]:
sampled_df['closest_to_deepl'] = sampled_df.apply(closest_to_deepl, axis=1)

In [ ]:
print("🎯 Distribution vs Expert (DeepL):")
print(sampled_df['closest_to_deepl'].value_counts(normalize=True).round(3))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity_columns(df, reference_col='translated_deepl'):
    """
    Gemini, Helsinki, M2M çevirilerini DeepL referansına göre karşılaştırır.
    Sonuçları sim_gemini, sim_helsinki, sim_m2m olarak ekler.
    """
    models = {
        'sim_gemini': 'translated_text',
        'sim_helsinki': 'helsinki_translation',
        'sim_m2m': 'translated_m2m'
    }

    for sim_col, model_col in models.items():
        sims = []
        for a, b in zip(df[model_col], df[reference_col]):
            if pd.isna(a) or pd.isna(b):
                sims.append(-1)
            else:
                vect = TfidfVectorizer().fit([a, b])
                tfidf = vect.transform([a, b])
                sim_score = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
                sims.append(sim_score)
        df[sim_col] = sims

    return df


In [ ]:
sampled_df = compute_similarity_columns(sampled_df)

In [ ]:
sampled_df['best_match'] = sampled_df.apply(
    lambda row: compare_similarity_three(
        row['sim_gemini'],
        row['sim_helsinki'],
        row['sim_m2m']
    ),
    axis=1
)

In [ ]:
print(sampled_df['best_match'].value_counts(dropna=False))


In [ ]:
print(sampled_df['best_match'].value_counts(normalize=True, dropna=False).round(3))


## tüm çevirileri tekrar skorla sıfırdan! işler karıştı. 